#Goal
1. To create a more efficient tokenizer than baseline which is able to preserve the heirachy of expressions
2. The output of equivalent expressions should be the same
3. Prevent the vocabulary from being filled with unnecessary tokens


In [ ]:
import numpy as np
import pandas as pd
import os
import re
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Dataset Building

In [ ]:
def splitter(text):
  parts = text.rsplit(":", 2) # assuming : doesnt appear in the amp and sq amp
  if len(parts) != 3:
    return None, None, None
  interaction = parts[0] #interaction plus feymnman diagram
  amp = parts[1].strip()
  sq_amp = parts[2].strip()
  return interaction, amp, sq_amp

In [ ]:
def build_corpus(data_directory):

    corpus = []

    if not os.path.exists(data_directory):
        print(f"Error: Colab cannot find a folder at '{data_directory}'")
        print("Check your folder name and path!")
        return corpus

    print(f"Scanning folder: {data_directory}...")

    for filename in os.listdir(data_directory):
        if filename.endswith(".txt"):
            filepath = os.path.join(data_directory, filename)

            with open(filepath, 'r', encoding='utf-8') as file:
                for line in file:
                    interaction, amp, sq_amp = splitter(line)

                    if amp and sq_amp:
                        corpus.append(amp)
                        corpus.append(sq_amp)

    print(f"Success, Corpus built with {len(corpus)} total mathematical expressions.")
    return corpus

In [ ]:
drive_folder_path = '/content/drive/MyDrive/Symba'
my_corpus = build_corpus(drive_folder_path)
if my_corpus:
    print("\nSample Equation 1:", my_corpus[0][:100], "...")
    print("Sample Equation 2:", my_corpus[1][:100], "...")

Scanning folder: /content/drive/MyDrive/Symba...
Success, Corpus built with 1188 total mathematical expressions.

Sample Equation 1: -1/2*i*e^2*gamma_{+%\tau_157,%gam_147,%eta_108}*gamma_{%\tau_157,%gam_148,%gam_149}*e_{l_3,%gam_147} ...
Sample Equation 2: 1/4*e^4*(16*m_e^2*m_t^2 + 8*m_e^2*s_12 + 8*s_14*s_23 + 8*s_13*s_24 + 8*m_t^2*s_34)*(m_t^2 + s_12 + 1 ...


# Standardizing
####(as taken from the baseline)

In [ ]:
import re

def perfect_standardizer(expression):
    """
    Uses '%' logic to perfectly identify and standardize
    dummy variables
    """
    mapping = {}
    dummy_counter = 1

    def replacer(match):
        nonlocal dummy_counter
        # Group 1 captures the prefix, e.g., "%\sigma_" or "%\mu_"
        prefix = match.group(1)
        # Group 2 captures the actual number, e.g., "18923"
        index_str = match.group(2)

        # RULE 1: Have we mapped this dummy number before?
        if index_str in mapping:
            return f"{prefix}{mapping[index_str]}"

        # RULE 2: Brand new dummy number = Assign it a 'd' label.
        new_id = f"d{dummy_counter}"
        mapping[index_str] = new_id
        dummy_counter += 1

        return f"{prefix}{new_id}"

    # THE REGEX:
    # Looks for a '%' sign, then any letters/symbols, then an '_', then a number.
    # It strictly ONLY triggers if the '%' sign is present.
    standardized_expr = re.sub(r'(%[a-zA-Z\\]+_)(\d+)', replacer, expression)

    return standardized_expr

#TEST
test_eq = "s_12 * gamma_{%\\sigma_18923} * p_1 + delta_{%\\mu_99} * gamma_{%\\sigma_18923}"

print("Original:", test_eq)
print("Cleaned: ", perfect_standardizer(test_eq))

Original: s_12 * gamma_{%\sigma_18923} * p_1 + delta_{%\mu_99} * gamma_{%\sigma_18923}
Cleaned:  s_12 * gamma_{%\sigma_d1} * p_1 + delta_{%\mu_d2} * gamma_{%\sigma_d1}


In [ ]:
standardized_corpus = [perfect_standardizer(eq) for eq in my_corpus]

# VALIDATION CHECK
print("VALIDATION ")
for i in range(2):
    print(f"Original {i+1}:", my_corpus[i])
    print(f"Cleaned  {i+1}:", standardized_corpus[i])
    print("-")

VALIDATION 
Original 1: -1/2*i*e^2*gamma_{+%\tau_157,%gam_147,%eta_108}*gamma_{%\tau_157,%gam_148,%gam_149}*e_{l_3,%gam_147}(p_3)_u^(*)*e_{j_5,%eta_108}(p_4)_v*t_{i_3,%gam_149}(p_1)_u*t_{k_3,%gam_148}(p_2)_v^(*)/(m_t^2 + s_12 + 1/2*reg_prop)
Cleaned  1: -1/2*i*e^2*gamma_{+%\tau_d1,%gam_d2,%eta_d3}*gamma_{%\tau_d1,%gam_d4,%gam_d5}*e_{l_3,%gam_d2}(p_3)_u^(*)*e_{j_5,%eta_d3}(p_4)_v*t_{i_3,%gam_d5}(p_1)_u*t_{k_3,%gam_d4}(p_2)_v^(*)/(m_t^2 + s_12 + 1/2*reg_prop)
-
Original 2: 1/4*e^4*(16*m_e^2*m_t^2 + 8*m_e^2*s_12 + 8*s_14*s_23 + 8*s_13*s_24 + 8*m_t^2*s_34)*(m_t^2 + s_12 + 1/2*reg_prop)^(-2)
Cleaned  2: 1/4*e^4*(16*m_e^2*m_t^2 + 8*m_e^2*s_12 + 8*s_14*s_23 + 8*s_13*s_24 + 8*m_t^2*s_34)*(m_t^2 + s_12 + 1/2*reg_prop)^(-2)
-


# Defining Tree Grammar
Tree representation allows us to get rid for multiple brackets while preserving heirachy of operations

In [ ]:
import re
!pip install lark
from lark import Lark, Transformer
#Grammar
qft_prefix_grammar = r"""
    ?start: expr

    ?expr: term (ADD_OP term)*
    ?term: factor (MUL_OP factor)*
    ?factor: "-" factor -> neg_node
           | "+" factor -> pos_node
           | power

    ?power: app ("^" power_arg)?
    ?power_arg: app
              | "(*)" -> conj_node

    // --- THE NEW LAYER: Binary Function Application (@) ---
    ?app: atom (APP_OP atom)*

    ?atom: base ("_" sub_arg)?
    ?sub_arg: base

    ?comma_list: expr (COMMA expr)*

    ?base: SYMBOL             -> var
         | NUMBER             -> num
         | "(" expr ")"       -> drop_group
         | "{" comma_list "}" -> drop_group

    ADD_OP: "+" | "-"
    MUL_OP: "*" | "/"
    APP_OP: "@"             // New explicit token!
    COMMA:  ","

    SYMBOL: /%?\\?[a-zA-Z][a-zA-Z0-9]*/
    NUMBER: /\d+(\.\d+)?/

    %import common.WS
    %ignore WS
"""

parser = Lark(qft_prefix_grammar, parser='lalr')

class QFTPrefixTransformer(Transformer):
    def _binary_flatten(self, args):
        if len(args) == 1: return args[0]
        res = [str(args[1])] + args[0] + args[2]
        for i in range(3, len(args), 2):
            res = [str(args[i])] + res + args[i+1]
        return res

    def expr(self, args): return self._binary_flatten(args)
    def term(self, args): return self._binary_flatten(args)
    def app(self, args): return self._binary_flatten(args)  # Handles the @ operator
    def comma_list(self, args): return self._binary_flatten(args)

    def power(self, args):
        if len(args) == 1: return args[0]
        return ["^"] + args[0] + args[1]

    def atom(self, args):
        if len(args) == 1: return args[0]
        return ["_"] + args[0] + args[1]

    def neg_node(self, args): return ["NEG"] + args[0]
    def pos_node(self, args): return ["UADD"] + args[0]
    def conj_node(self, args): return ["CONJ"]

    def drop_group(self, args): return args[0]
    def var(self, args): return [str(args[0])]
    def num(self, args): return [str(args[0])]

transformer = QFTPrefixTransformer()
print("SUCCESSSSS")

SUCCESSSSS


In [ ]:
import time
import re

def preprocess_equation(equation):
    """
    Cleans up physics shorthands into strict mathematical syntax for the AST.
    """
    # Rule 1 & 2: Implicit Application (e.g., f(x) -> f @ (x))
    eq = re.sub(r'([\}\)])([\(])', r'\1 @ \2', equation)
    eq = re.sub(r'([a-zA-Z0-9])([\(])', r'\1 @ \2', eq)

    # --- RULE 3 FIXED: Grouping Chained Subscripts
    # Example: p_4_+%\sigma_d2 => {p_4}_{+%\sigma_d2}
    eq = re.sub(r'([a-zA-Z]_\d+)_([+-]?(?:%?\\?[a-zA-Z]+)_[a-zA-Z0-9]+)', r'{\1}_{\2}', eq)

    # Rule 4: Compound Tensor Names
    # Example: T_C_10_{...}  => {{{T}_{C}}_{10}}_{...}
    eq = re.sub(r'([a-zA-Z]+)_([a-zA-Z]+)_(\d+)(_\{)', r'{{{\1}_{\2}}_{\3}}\4', eq)
    eq = re.sub(r'(?<![A-Za-z\}])([a-zA-Z]+)_([a-zA-Z0-9]+)(_\{)', r'{{\1}_{\2}}\3', eq)

    return eq

def qft_to_prefix(equation):
    processed_eq = preprocess_equation(equation)
    tree = parser.parse(processed_eq)
    return transformer.transform(tree)

#FULL CORPUS PROCESSING
final_ai_dataset = []
failed_equations = []

print(f"Starting parsing for {len(standardized_corpus)} equations...")
start_time = time.time()

for i, equation in enumerate(standardized_corpus):
    try:
        prefix_tokens = qft_to_prefix(equation)
        final_ai_dataset.append(" ".join(prefix_tokens))
    except Exception as e:
        failed_equations.append((i, equation, str(e)))

# Save to text file
with open("qed_prefix_training_data.txt", "w") as f:
    for eq in final_ai_dataset:
        f.write(f"{eq}\n")

print(f"\nPROCESSING COMPLETE")
print(f"Successfully Parsed : {len(final_ai_dataset)}")
print(f"Failed to Parse     : {len(failed_equations)}")
print(f"Time Taken          : {time.time() - start_time:.2f} seconds")

if failed_equations:
    print("\n--- TOP FAILED EQUATIONS ---")
    for fail in failed_equations[:3]:
        print(f"Index {fail[0]}: {fail[1]}")
        print(f"Error: {fail[2]}\n")

Starting parsing for 1188 equations...

PROCESSING COMPLETE
Successfully Parsed : 1188
Failed to Parse     : 0
Time Taken          : 5.37 seconds


In [ ]:
import json
from collections import Counter

with open("qed_prefix_training_data.txt", "r") as f:
    equations = f.readlines()

all_tokens = []
equation_lengths = []

for eq in equations:
    raw_tokens = eq.strip().split()
    processed_tokens = []

    for token in raw_tokens:
        # If it is a pure number (like "5839" or "-2"), break it into digits
        if token.isdigit() or (token.startswith('-') and token[1:].isdigit()):
            if token.startswith('-'):
                processed_tokens.append('-')
                processed_tokens.extend(list(token[1:]))
            else:
                processed_tokens.extend(list(token))
        else:
            # Leave variables (like T_C_10 or p_4) and operators as single tokens
            processed_tokens.append(token)

    all_tokens.extend(processed_tokens)
    equation_lengths.append(len(processed_tokens))

token_counts = Counter(all_tokens)

# the foundational math characters -
vocab2id = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<SOS>": 2,
    "<EOS>": 3,
    "-": 4,
    "0": 5, "1": 6, "2": 7, "3": 8, "4": 9,
    "5": 10, "6": 11, "7": 12, "8": 13, "9": 14
}

# adding everything else (like APP_OP, T_C_10) starting from ID 15
current_id = 15
for token, count in token_counts.items():
    if token not in vocab2id:
        vocab2id[token] = current_id
        current_id += 1

id2vocab = {v: k for k, v in vocab2id.items()}

with open("vocab2id.json", "w") as f:
    json.dump(vocab2id, f)

with open("id2vocab.json", "w") as f:
    json.dump(id2vocab, f)

print("\n VOCABULARY REBUILT WITH FIXED DIGITS ")
print(f"New AI Vocab Size   : {len(vocab2id)}")
print(f"Longest eq length   : {max(equation_lengths)} tokens")


 VOCABULARY REBUILT WITH FIXED DIGITS 
New AI Vocab Size   : 181
Longest eq length   : 2859 tokens


In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


with open("vocab2id.json", "r") as f:
    vocab2id = json.load(f)

# special tokens
PAD_IDX = vocab2id["<PAD>"]
SOS_IDX = vocab2id["<SOS>"]
EOS_IDX = vocab2id["<EOS>"]
UNK_IDX = vocab2id["<UNK>"]

def tokenize_and_convert(equation_string):
    """Applies the exact same digit-splitting logic and converts to IDs."""
    raw_tokens = equation_string.strip().split()
    processed_ids = [SOS_IDX] # Always start with <SOS>

    for token in raw_tokens:
        if token.isdigit() or (token.startswith('-') and token[1:].isdigit()):
            if token.startswith('-'):
                processed_ids.append(vocab2id.get('-', UNK_IDX))
                for digit in token[1:]:
                    processed_ids.append(vocab2id.get(digit, UNK_IDX))
            else:
                for digit in token:
                    processed_ids.append(vocab2id.get(digit, UNK_IDX))
        else:
            processed_ids.append(vocab2id.get(token, UNK_IDX))

    processed_ids.append(EOS_IDX) # Always end with <EOS>
    return torch.tensor(processed_ids, dtype=torch.long)

# dataset building
class QEDDataset(Dataset):
    def __init__(self, file_path):
        with open(file_path, "r") as f:
            equations = f.readlines()

        # Convert all text equations to integer tensors
        print("Converting text to integer tensors...")
        self.data = [tokenize_and_convert(eq) for eq in equations]

        # YOUR FIX: Sort the dataset by length so batches have similarly sized equations!
        self.data.sort(key=lambda x: len(x))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Dynamic Padding
def dynamic_pad_collate(batch):
    """
    Takes a batch of tensors of varying lengths, and pads them ONLY
    to the length of the longest tensor in this specific batch.
    """
    # pad_sequence finds the max length in batch and pads with PAD_IDX
    padded_batch = pad_sequence(batch, batch_first=True, padding_value=PAD_IDX)
    return padded_batch


# Initialize the dataset
dataset = QEDDataset("qed_prefix_training_data.txt")
print(f"Total equations loaded: {len(dataset)}")

# Create the DataLoader
# batch_size=16 means we process 16 equations at a time.
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=dynamic_pad_collate
)

print("\n BATCH SHAPE TEST")
#test
batches = list(dataloader)

print(f"Batch 1 (Shortest equations) Shape : {batches[0].shape}")
print(f"Batch 2 Shape                      : {batches[1].shape}")
print(f"Batch 3 Shape                      : {batches[2].shape}")
print(f"Last Batch (The 2666 monster) Shape: {batches[-1].shape}")

Converting text to integer tensors...
Total equations loaded: 1188

 BATCH SHAPE TEST
Batch 1 (Shortest equations) Shape : torch.Size([16, 96])
Batch 2 Shape                      : torch.Size([16, 97])
Batch 3 Shape                      : torch.Size([16, 97])
Last Batch (The 2666 monster) Shape: torch.Size([4, 2861])


In [ ]:
import random

# 1. Load the raw text
with open("qed_prefix_training_data.txt", "r") as f:
    equations = f.readlines()

# 2. Shuffle the dataset randomly (Crucial: Set a seed for reproducibility!)
random.seed(42)
random.shuffle(equations)

# 3. Calculate the 80-10-10 split indices
total_eqs = len(equations)
train_split = int(0.8 * total_eqs)
val_split = train_split + int(0.1 * total_eqs)

# 4. Slice the lists
train_data = equations[:train_split]
val_data = equations[train_split:val_split]
test_data = equations[val_split:]

print(f"Total Equations : {total_eqs}")
print(f"Training Set    : {len(train_data)} (80%)")
print(f"Validation Set  : {len(val_data)} (10%)")
print(f"Test Set        : {len(test_data)} (10%)")

# Next Step: You would feed 'train_data', 'val_data', and 'test_data'
# individually into your QEDDataset class so they sort themselves internally!

Total Equations : 1188
Training Set    : 950 (80%)
Validation Set  : 118 (10%)
Test Set        : 120 (10%)


# Outcome

## Structural Improvements Achieved

1. Mathematical expressions were successfully converted into explicit hierarchical representations using grammar-aware parsing and Abstract Syntax Trees (ASTs).

2. Prefix serialisation eliminated bracket dependency while still preserving compositional and operator-level structure.

3. Structurally equivalent expressions now map to significantly more consistent token sequences, improving symbolic alignment between input and output representations.

4. Vocabulary quality improved substantially by replacing frequency-driven fragmentation with deterministic structure-aware token construction.


## Key Insight

Unlike natural language, symbolic mathematics is governed by strict compositional and structural rules. As a result, preserving hierarchy and operator semantics proved more important than optimizing purely statistical token frequency.

This refined pipeline established the foundation required for the downstream squared-amplitude prediction model.